# DeepGuard — DF40 + DeepfakeBench + LiR (Google Drive workspace)

This notebook uses Google Drive for persistent storage and Colab GPU for computation. It is resumable: detector outputs, manifests, hashes and results are stored on Drive.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
ROOT=Path('/content/drive/MyDrive/DeepGuard')
for p in ['datasets','models','features','lir/development','lir/calibration','lir/validation','reports','logs','manifests']:
    (ROOT/p).mkdir(parents=True,exist_ok=True)
print('DeepGuard workspace:',ROOT)


In [ ]:
!nvidia-smi || true
!df -h /content /content/drive | tail -n +1
!python --version


In [ ]:
%cd /content
import os
if not os.path.exists('/content/deepguard-forensic-lr'):
  !git clone https://github.com/geradts/deepguard-forensic-lr.git
!pip install -q -r /content/deepguard-forensic-lr/requirements.txt


## 1. Persistent environment manifest


In [ ]:
import subprocess, json, platform, datetime
env={
 'utc':datetime.datetime.now(datetime.timezone.utc).isoformat(),
 'python':platform.python_version(),
 'platform':platform.platform(),
 'git_deepguard':subprocess.getoutput('git -C /content/deepguard-forensic-lr rev-parse HEAD'),
 'gpu':subprocess.getoutput('nvidia-smi --query-gpu=name,memory.total --format=csv,noheader 2>/dev/null')
}
(ROOT/'logs/environment.json').write_text(json.dumps(env,indent=2))
print(json.dumps(env,indent=2))


## 2. Upstream repositories

These are kept outside the GitHub project and their licenses/terms remain applicable. Large data should be stored on Drive, not in the Git repository.


In [ ]:
from pathlib import Path
for repo,url in [('DeepfakeBench','https://github.com/SCLBD/DeepfakeBench.git'),('DF40','https://github.com/YZY-stack/DF40.git')]:
    target=Path('/content')/repo
    if not target.exists():
        !git clone {url} {target}
    print(repo, 'ready')


## 3. Dataset manifest

Do not put copyrighted/large video files into GitHub. Store their paths and hashes in Drive.


In [ ]:
import pandas as pd
manifest=ROOT/'manifests/df40_pilot_manifest.csv'
if manifest.exists():
    df=pd.read_csv(manifest); display(df.head()); print(len(df),'rows')
else:
    print('Create the manifest at:',manifest)
    print('Required columns: video_id,label,source_id,subject_id,generator_id,dataset,path')


## 4. Detector checkpoints

Place the official DeepfakeBench checkpoints in Drive under `models/`. The notebook does not fabricate or redistribute weights.


In [ ]:
print('Expected persistent model location:',ROOT/'models')
print('Keep checkpoint SHA-256 hashes in the experiment manifest.')


## 5. Resumable detector stage

Run Xception/FTCN using the repository scripts/configurations. Save each detector's output CSV to Drive. Existing output files should be reused rather than recomputed.


In [ ]:
print('Detector stage prepared.')
print('Outputs:',ROOT/'features/xception', ROOT/'features/ftcn')


## 6. Feature fusion


In [ ]:
print('Next: merge detector outputs with source/subject/generator metadata into a frozen feature table.')
print('Target:',ROOT/'features/fused/deepguard_features.csv')


## 7. LiR

Only after the development/calibration split is frozen should external validation be opened.


In [ ]:
print('LiR hand-off:',ROOT/'lir')
print('Use the versioned lir_handoff.yaml from the repository.')


## 8. Resume / audit

At every stage save timestamps, git commits, checkpoint hashes, dataset manifest hashes, software versions and outputs.


In [ ]:
import hashlib
def sha256(path):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for b in iter(lambda:f.read(1024*1024),b''): h.update(b)
    return h.hexdigest()
print('Workspace ready:',ROOT)
